#**This notebook implements all the versions of the allosteric pathway - finding mentioned in the manuscript.**

#**Each version requires inputs in slightly different formats. This is specified below in each section.**

#**Initialisation - 1**
The cell below initialises all functions and objects required for the first three algorithms.

In [ ]:
import random, warnings
from collections import defaultdict
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
from Bio.PDB import PDBParser

CONTACT_CUTOFF   = 3.4
HOTSPOT_CUTOFF   = 4.5
DEFAULT_ALPHA    = 3.0
DEFAULT_N_ROUNDS = 10_000

def build_contact_matrix(residue_atoms, backbone_idx, cutoff=CONTACT_CUTOFF):
    N, cutoff_sq = len(residue_atoms), cutoff ** 2
    C = np.zeros((N, N), dtype=np.int32)
    for i in range(N):
        ci = residue_atoms[i]
        for j in range(i + 1, N):
            cj   = residue_atoms[j]
            diff = ci[:, None, :] - cj[None, :, :]
            mask = (diff * diff).sum(axis=2) <= cutoff_sq
            if abs(i - j) == 1:
                bb_i = np.array(sorted(backbone_idx[i]), dtype=np.intp)
                bb_j = np.array(sorted(backbone_idx[j]), dtype=np.intp)
                if bb_i.size and bb_j.size:
                    mask[np.ix_(bb_i, bb_j)] = False
            C[i, j] = C[j, i] = int(mask.sum())
    return C

def average_contacts(C, atom_counts):
    eps   = 1e-12
    N_fwd = C / (atom_counts[:, None].astype(float) + eps)
    N_rev = C / (atom_counts[None, :].astype(float) + eps)
    return N_fwd, N_rev

def propagation_matrix(N_fwd, alpha=DEFAULT_ALPHA):
    return 1.0 - np.exp(-alpha * N_fwd)

def normalize_tc_matrix(TC, method="minmax", floor=1e-2, invert=False):
    TC = (np.array(TC, dtype=np.float64) + TC.T) / 2.0

    if method == "minmax":
        lo, hi = TC.min(), TC.max()
        TC_norm = np.ones_like(TC) if hi - lo < 1e-12 else (TC - lo) / (hi - lo)
    elif method == "sigmoid":
        finite_mask = np.isfinite(TC)
        mu  = TC[finite_mask].mean()
        sig = TC[finite_mask].std() + 1e-12
        TC_norm = np.where(finite_mask,
                          1.0 / (1.0 + np.exp(-(TC - mu) / sig)),
                          1.0)
    elif method == "rank":
        flat    = TC.flatten()
        TC_norm = flat.argsort().argsort().reshape(TC.shape).astype(np.float64)
        TC_norm /= TC_norm.max() + 1e-12
    elif method == "NO":
        TC_norm = TC
    else:
        raise ValueError(f"Unknown method '{method}'. Use 'minmax', 'sigmoid', or 'rank'.")

    if invert:
        TC_norm = 1.0 - TC_norm

    return floor + (1.0 - floor) * TC_norm

def propagation_matrix_with_tc(N_fwd, TC, alpha=DEFAULT_ALPHA,
                                tc_weight=1.0, tc_norm_method="minmax",
                                tc_invert=False):
    assert N_fwd.shape == TC.shape, f"{N_fwd.shape} {TC.shape}"
    TC_norm  = normalize_tc_matrix(TC, method=tc_norm_method, invert=tc_invert)
    alpha_ij = alpha * (TC_norm ** tc_weight)
    return 1.0 - np.exp(-alpha_ij * N_fwd)


class OhmAnalyzer:

    def __init__(self, P, residue_labels=None, coords=None):
        self.P              = P.astype(np.float64)
        self.N              = P.shape[0]
        self.residue_labels = residue_labels or [str(i) for i in range(self.N)]
        self.coords         = coords
        self._nbrs          = [np.where(P[i] > 0)[0].tolist() for i in range(self.N)]

    @classmethod
    def _parse_pdb(cls, pdb_path, chain_ids=None, cutoff=CONTACT_CUTOFF):
        BACKBONE = {"N", "CA", "C", "O"}
        parser   = PDBParser(QUIET=True)
        model    = next(parser.get_structure("prot", pdb_path).get_models())
        res_coords, res_bb_idx, labels, ca_coords = [], [], [], []
        for chain in model:
            if chain_ids and chain.id not in chain_ids:
                continue
            for res in chain:
                het, seqid, _ = res.get_id()
                if het.strip(): continue
                atoms = list(res.get_atoms())
                if not atoms: continue
                coords = np.array([a.get_vector().get_array() for a in atoms])
                bb_set = {k for k, a in enumerate(atoms) if a.name in BACKBONE}
                ca     = next((a for a in atoms if a.name == "CA"), atoms[0])
                res_coords.append(coords)
                res_bb_idx.append(bb_set)
                labels.append(f"{chain.id}/{seqid}")
                ca_coords.append(ca.get_vector().get_array())
        atom_counts = np.array([len(c) for c in res_coords], dtype=np.int32)
        C = build_contact_matrix(res_coords, res_bb_idx, cutoff)
        N_fwd, _ = average_contacts(C, atom_counts)

        import pathlib
        _stem = pathlib.Path(pdb_path).stem
        np.savetxt(f"{_stem}_contact_coupling.csv", N_fwd, delimiter=",", fmt="%.6f")
        print(f"Saved: {_stem}_contact_matrix_raw.npy / .csv")
        print(f"Saved: {_stem}_contact_matrix_norm.npy / .csv")

        return N_fwd, labels, np.array(ca_coords)

    @classmethod
    def from_pdb(cls, pdb_path, chain_ids=None, alpha=DEFAULT_ALPHA,
                 cutoff=CONTACT_CUTOFF):
        N_fwd, labels, coords = cls._parse_pdb(pdb_path, chain_ids, cutoff)
        return cls(propagation_matrix(N_fwd, alpha), residue_labels=labels, coords=coords)

    @classmethod
    def from_pdb_with_tc(cls, pdb_path, TC, chain_ids=None,
                         alpha=DEFAULT_ALPHA, cutoff=CONTACT_CUTOFF,
                         tc_weight=1.0, tc_norm_method="minmax", tc_invert=False):
        N_fwd, labels, coords = cls._parse_pdb(pdb_path, chain_ids, cutoff)
        P = propagation_matrix_with_tc(N_fwd, TC, alpha=alpha,
                                        tc_weight=tc_weight,
                                        tc_norm_method=tc_norm_method,
                                        tc_invert=tc_invert)
        return cls(P, residue_labels=labels, coords=coords)

    def compute_aci(self, ACTIVE_RESNUMSs, n_rounds=DEFAULT_N_ROUNDS, seed=None):
        rng, P, N, nbrs = random.Random(seed), self.P, self.N, self._nbrs
        source, T       = list(ACTIVE_RESNUMSs), np.zeros(N)
        for _ in range(n_rounds):
            V, W, queue = [0]*N, [0]*N, []
            for s in source:
                V[s] = W[s] = 1; queue.append(s)
            head = 0
            while head < len(queue):
                i = queue[head]; head += 1
                for j in nbrs[i]:
                    if not W[j]:
                        W[j] = 1
                        if rng.random() < P[i, j]: V[j] = 1
                        queue.append(j)
            for i in range(N): T[i] += V[i]
        return T / n_rounds

    def find_hotspots(self, aci, exclude_residues=None,
                      distance_threshold=HOTSPOT_CUTOFF):
        if self.coords is None:
            raise ValueError("Cα coordinates required. Use from_pdb().")
        exclude   = set(exclude_residues or [])
        N, coords = self.N, self.coords
        direction = np.full(N, -1, dtype=np.int32)
        for i in range(N):
            if i in exclude: continue
            best_val, best_j = aci[i], -1
            for j in range(N):
                if j == i or j in exclude: continue
                if np.linalg.norm(coords[i]-coords[j]) <= distance_threshold:
                    if aci[j] > best_val: best_val, best_j = aci[j], j
            direction[i] = best_j
        memo = {}
        def centre(i):
            if i in memo: return memo[i]
            memo[i] = i if direction[i]==-1 else centre(direction[i])
            return memo[i]
        clusters = defaultdict(list)
        for i in range(N):
            if i not in exclude: clusters[centre(i)].append(i)
        return sorted(clusters.values(), key=lambda c: max(aci[r] for r in c), reverse=True)

    def find_pathways(self, active_site, allosteric_site,
                      n_rounds=DEFAULT_N_ROUNDS, seed=None, max_depth=30):
        rng            = random.Random(seed)
        P, nbrs        = self.P, self._nbrs
        active_set     = set(active_site)
        allosteric_set = set(allosteric_site)
        path_counts, total = defaultdict(int), 0
        for _ in range(n_rounds):
            current = rng.choice(list(active_site))
            path, visited = [-1, current], {current}
            for _ in range(max_depth):
                cands = [j for j in nbrs[current] if j not in visited]
                if not cands: break
                weights = [P[current, j] for j in cands]
                total_w = sum(weights)
                if total_w == 0: break
                r, acc  = rng.random() * total_w, 0.0
                next_r  = cands[-1]
                for j, w in zip(cands, weights):
                    acc += w
                    if r <= acc: next_r = j; break
                path.append(next_r); visited = visited | {next_r}; current = next_r
                if current in allosteric_set:
                    path.append(-2); path_counts[tuple(path)] += 1; total += 1; break
        if total == 0:
            warnings.warn(f"no pathways found. might have to increase max_depth (current={max_depth}).")
            return {}
        return {p: c/total for p, c in path_counts.items()}

    def critical_residues(self, path_weights):
        importance = defaultdict(float)
        for path, w in path_weights.items():
            for r in path:
                if r < 0: continue
                pa = importance[r]
                importance[r] = pa + w - pa * w
        return dict(importance)

    def aci_report(self, aci, top_n=20):
        ranked = np.argsort(aci)[::-1]
        lines  = [f"{'Rank':<6} {'Residue':<12} {'ACI':>8}"]
        for k, idx in enumerate(ranked[:top_n], 1):
            lines.append(f"{k:<6} {self.residue_labels[idx]:<12} {aci[idx]:>8.4f}")
        return "\n".join(lines)


    def pathway_report(self, path_weights, top_n=10):
        lines = [f"{'Pathway':<65} {'Weight':>8}"]
        for path, w in sorted(path_weights.items(), key=lambda x: x[1], reverse=True)[:top_n]:
            names = ["Start" if r==-1 else "End" if r==-2 else self.residue_labels[r] for r in path]
            lines.append(f"{' -> '.join(names):<65} {w:>8.4f}")
        return "\n".join(lines)
        print('')

    def critical_residue_report(self, path_weights, top_n=15):
        imp   = self.critical_residues(path_weights)
        lines = [f"{'Rank':<6} {'Residue':<12} {'Importance':>12}"]
        for k, (idx, score) in enumerate(sorted(imp.items(), key=lambda x: x[1], reverse=True)[:top_n], 1):
            lines.append(f"{k:<6} {self.residue_labels[idx]:<12} {score:>12.4f}")
        return "\n".join(lines)
        print('')

    @classmethod
    def from_pdb_with_tc_stored(cls, pdb_path, TC, chain_ids=None,
                                 alpha=DEFAULT_ALPHA, cutoff=CONTACT_CUTOFF):
        N_fwd, labels, coords = cls._parse_pdb(pdb_path, chain_ids, cutoff)
        assert TC.shape == N_fwd.shape, f"TC shape {TC.shape} must match residue count {N_fwd.shape}"
        P   = propagation_matrix(N_fwd, alpha)
        obj = cls(P, residue_labels=labels, coords=coords)
        obj.TC = np.array(TC, dtype=np.float64)
        return obj

    def compute_aci_tc_filtered(self, ACTIVE_RESNUMSs,
                                 n_rounds=DEFAULT_N_ROUNDS,
                                 seed=None,
                                 top_k=10):

        rng, P, TC = random.Random(seed), self.P, self.TC
        N           = self.N
        source      = list(ACTIVE_RESNUMSs)
        T           = np.zeros(N)

        tc_nbrs = []
        for i in range(N):
            structural_nbrs = self._nbrs[i]
            if len(structural_nbrs) <= top_k:
                tc_nbrs.append(structural_nbrs)
            else:
                tc_scores = np.abs(TC[i, structural_nbrs])
                ranked    = np.argsort(tc_scores)[::-1][:top_k]
                tc_nbrs.append([structural_nbrs[r] for r in ranked])

        for _ in range(n_rounds):
            V, W, queue = [0]*N, [0]*N, []
            for s in source:
                V[s] = W[s] = 1; queue.append(s)
            head = 0
            while head < len(queue):
                i = queue[head]; head += 1
                for j in tc_nbrs[i]:
                    if not W[j]:
                        W[j] = 1
                        if rng.random() < P[i, j]:
                            V[j] = 1
                        queue.append(j)
            for i in range(N):
                T[i] += V[i]

        return T / n_rounds


print("ran successfully")

ran successfully


In [ ]:
!pip install biopython numpy scipy -q

# **Base algorithm (Wang et al., 2020)**

Inputs:
1) Protein structure in PDB format.
2) Known active and allosteric residue numbers for the protein.

Outputs:

1.   Allosteric Coupling Intensity ranks for residues (Top 20)
2.   Allosteric Hotspots (Top 5)
3.   Allosteric Pathways (Top 5)
4.   Importance rank for residues (Top 15)




In [ ]:
import urllib.request

pdb_path = 'Rhodopsin.pdb'
ohm = OhmAnalyzer.from_pdb(pdb_path)
"""
ACTIVE_RESNUMS = [113, 122, 126, 178, 181, 191, 207, 208, 212, 211,
                        264, 265, 267, 268, 269, 293, 296]
ALLOSTERIC_RESNUMS = [134, 135, 136, 247, 251, 55, 83, 298, 299, 302, 303, 306, 313]


#Villin
ACTIVE_RESNUMS     = [64, 70, 71, 73, 75, 76]
ALLOSTERIC_RESNUMS = [47, 50, 51, 52, 53, 58, 61, 65, 69]
"""
#RAS
ACTIVE_RESNUMS = [10, 11, 12, 13, 14, 15, 16, 17, 33, 35, 36, 37, 38, 39, 40, 41]
ALLOSTERIC_RESNUMS = [57, 60, 61, 62, 64, 68, 72]


N_ROUNDS, SEED    = 10_000, 42

def get_idx(ohm, resnums):
    return [i for i, l in enumerate(ohm.residue_labels)
            if int(l.split("/")[1]) in resnums]

active_idx     = get_idx(ohm, ACTIVE_RESNUMS)
allosteric_idx = get_idx(ohm, ALLOSTERIC_RESNUMS)

aci      = ohm.compute_aci(active_idx, n_rounds=N_ROUNDS, seed=SEED)
print(ohm.aci_report(aci, top_n=20))
print('\n')

hotspots = ohm.find_hotspots(aci, exclude_residues=active_idx)
for k, cluster in enumerate(hotspots[:5], 1):
    print(f"Hotspot {k} (ACI={max(aci[r] for r in cluster):.4f}): "
          f"{[[ohm.residue_labels[r] for r in cluster]]}")
print('\n')

pathways = ohm.find_pathways(active_idx, allosteric_idx, n_rounds=N_ROUNDS, seed=SEED)
print(ohm.pathway_report(pathways, top_n=10))
print('\n')
print(ohm.critical_residue_report(pathways, top_n=15))
print('\n')

Saved: Rhodopsin_contact_matrix_raw.npy / .csv
Saved: Rhodopsin_contact_matrix_norm.npy / .csv
Rank   Residue           ACI
1      A/211          1.0000
2      A/264          1.0000
3      A/122          1.0000
4      A/113          1.0000
5      A/126          1.0000
6      A/293          1.0000
7      A/296          1.0000
8      A/267          1.0000
9      A/268          1.0000
10     A/269          1.0000
11     A/207          1.0000
12     A/178          1.0000
13     A/181          1.0000
14     A/212          1.0000
15     A/191          1.0000
16     A/208          1.0000
17     A/265          1.0000
18     A/344          0.9700
19     A/338          0.9358
20     A/328          0.9251


Hotspot 1 (ACI=0.9700): [['A/343', 'A/344', 'A/345']]
Hotspot 2 (ACI=0.9358): [['A/65', 'A/337', 'A/338', 'A/339']]
Hotspot 3 (ACI=0.9251): [['A/327', 'A/328', 'A/329']]
Hotspot 4 (ACI=0.9175): [['A/340', 'A/341', 'A/342']]
Hotspot 5 (ACI=0.8940): [['A/23', 'A/24', 'A/25', 'A/26']]


Pathway  

# **TC - integrated algorithm**

Inputs:
1) Thermodynamic coupling matrix (derived from the bWSME model).
2) Known active and allosteric residue numbers for the protein.

Outputs:

1.   Allosteric Coupling Intensity ranks for residues (Top 20)
2.   Allosteric Hotspots (Top 5)
3.   Allosteric Pathways (Top 5)
4.   Importance rank for residues (Top 15)




In [ ]:
import scipy.io

TC  = pd.read_csv("Rhodopsin_bWSME_coupling.csv", header = None)
print(TC.shape)
TC = TC.to_numpy()
np.fill_diagonal(TC, 0)

print(f"TC range: {TC.min():.3f} → {TC.max():.3f}")

ohm_tc = OhmAnalyzer.from_pdb_with_tc(
    "Rhodopsin.pdb",
    TC,
    tc_weight=1.0,
    tc_norm_method="sigmoid",
    tc_invert=True,
)

active_idx_tc     = get_idx(ohm_tc, ACTIVE_RESNUMS)
allosteric_idx_tc = get_idx(ohm_tc, ALLOSTERIC_RESNUMS)

aci_tc   = ohm_tc.compute_aci(active_idx_tc, n_rounds=N_ROUNDS, seed=SEED)
print(ohm_tc.aci_report(aci_tc, top_n=20))
print('\n')
hotspots_tc = ohm_tc.find_hotspots(aci_tc, exclude_residues=active_idx_tc)
for k, cluster in enumerate(hotspots_tc[:5], 1):
    print(f"Hotspot {k} (ACI={max(aci_tc[r] for r in cluster):.4f}): "
          f"{[[ohm_tc.residue_labels[r] for r in cluster]]}")
print('\n')
pathways_tc = ohm_tc.find_pathways(active_idx_tc, allosteric_idx_tc, n_rounds=N_ROUNDS, seed=SEED)
print(ohm_tc.pathway_report(pathways_tc, top_n=10))
print('\n')
print(ohm_tc.critical_residue_report(pathways_tc, top_n=15))
print('\n')

(348, 348)
TC range: -25.820 → 45.322
Saved: Rhodopsin_contact_matrix_raw.npy / .csv
Saved: Rhodopsin_contact_matrix_norm.npy / .csv
Rank   Residue           ACI
1      A/293          1.0000
2      A/296          1.0000
3      A/265          1.0000
4      A/264          1.0000
5      A/269          1.0000
6      A/268          1.0000
7      A/267          1.0000
8      A/126          1.0000
9      A/122          1.0000
10     A/113          1.0000
11     A/178          1.0000
12     A/191          1.0000
13     A/208          1.0000
14     A/181          1.0000
15     A/207          1.0000
16     A/212          1.0000
17     A/211          1.0000
18     A/24           0.7039
19     A/338          0.7023
20     A/285          0.6867


Hotspot 1 (ACI=0.7039): [['A/21', 'A/22', 'A/23', 'A/24', 'A/25']]
Hotspot 2 (ACI=0.7023): [['A/65', 'A/336', 'A/337', 'A/338', 'A/339']]
Hotspot 3 (ACI=0.6867): [['A/182', 'A/283', 'A/284', 'A/285', 'A/286']]
Hotspot 4 (ACI=0.6285): [['A/342', 'A/343', 'A

# **Filtered neighbor algorithm**

Inputs:
1) Protein structure in PDB format.
2) Known active and allosteric residue numbers for the protein.

Outputs:

1.   Allosteric Coupling Intensity ranks for residues (Top 20)
2.   Allosteric Hotspots (Top 5)
3.   Allosteric Pathways (Top 5)
4.   Importance rank for residues (Top 15)




In [ ]:
ohm_c = OhmAnalyzer.from_pdb_with_tc_stored(
    "Rhodopsin.pdb",
    TC,
)

active_idx_c     = get_idx(ohm_c, ACTIVE_RESNUMS)
allosteric_idx_c = get_idx(ohm_c, ALLOSTERIC_RESNUMS)

aci_c = ohm_c.compute_aci_tc_filtered(
    active_idx_c,
    n_rounds=N_ROUNDS,
    seed=SEED,
    top_k=10,
)

print(ohm_c.aci_report(aci_c, top_n=20))
print('\n')
hotspots_c = ohm_c.find_hotspots(aci_c, exclude_residues=active_idx_c)
for k, cluster in enumerate(hotspots_c[:5], 1):
    print(f"Hotspot {k} (ACI={max(aci_c[r] for r in cluster):.4f}): "
          f"{[[ohm_c.residue_labels[r] for r in cluster]]}")
print('\n')
pathways_c = ohm_c.find_pathways(active_idx_c, allosteric_idx_c,
                                   n_rounds=N_ROUNDS, seed=SEED)
print(ohm_c.pathway_report(pathways_c, top_n=10))
print('\n')
print(ohm_c.critical_residue_report(pathways_c, top_n=15))
print('\n')

Saved: Rhodopsin_contact_matrix_raw.npy / .csv
Saved: Rhodopsin_contact_matrix_norm.npy / .csv
Rank   Residue           ACI
1      A/212          1.0000
2      A/211          1.0000
3      A/178          1.0000
4      A/191          1.0000
5      A/126          1.0000
6      A/122          1.0000
7      A/113          1.0000
8      A/264          1.0000
9      A/296          1.0000
10     A/265          1.0000
11     A/269          1.0000
12     A/268          1.0000
13     A/267          1.0000
14     A/181          1.0000
15     A/207          1.0000
16     A/208          1.0000
17     A/293          1.0000
18     A/344          0.9681
19     A/338          0.9339
20     A/328          0.9278


Hotspot 1 (ACI=0.9681): [['A/343', 'A/344', 'A/345']]
Hotspot 2 (ACI=0.9339): [['A/65', 'A/337', 'A/338', 'A/339']]
Hotspot 3 (ACI=0.9278): [['A/327', 'A/328', 'A/329']]
Hotspot 4 (ACI=0.9175): [['A/340', 'A/341', 'A/342']]
Hotspot 5 (ACI=0.8940): [['A/23', 'A/24', 'A/25', 'A/26']]


Pathway  

#**Initialiation - 2**
For the next two algorithms, the additional functions and classes required are defined below.


In [ ]:
import numpy as np
import pandas as pd

def normalize_dca_matrix(
    DCA,
    method: str = "minmax",
    floor: float = 1e-3,
    zero_negatives: bool = True,
) -> np.ndarray:
    DCA = (np.array(DCA, dtype=np.float64) + np.array(DCA, dtype=np.float64).T) / 2.0
    np.fill_diagonal(DCA, 0.0)

    if zero_negatives:
        DCA = np.where(DCA < 0, 0.0, DCA)

    if method == "minmax":
        lo, hi = DCA.min(), DCA.max()
        if hi - lo < 1e-12:
            P_norm = np.ones_like(DCA)
        else:
            P_norm = (DCA - lo) / (hi - lo)
    elif method == "sigmoid":
        mu  = DCA.mean()
        sig = DCA.std() + 1e-12
        P_norm = 1.0 / (1.0 + np.exp(-(DCA - mu) / sig))
    elif method == "rank":
        flat    = DCA.flatten()
        ranks   = flat.argsort().argsort().reshape(DCA.shape).astype(np.float64)
        P_norm  = ranks / (ranks.max() + 1e-12)
    else:
        raise ValueError(f"Unknown method '{method}'. Use 'minmax', 'sigmoid', or 'rank'.")

    return floor + (1.0 - floor) * P_norm


@classmethod
def _from_dca_matrix(
    cls,
    DCA,
    residue_labels,
    coords=None,
    dca_norm_method: str = "minmax",
    floor: float = 1e-3,
    zero_negatives: bool = True,
):
    P_dca = normalize_dca_matrix(
        DCA,
        method=dca_norm_method,
        floor=floor,
        zero_negatives=zero_negatives,
    )
    obj = cls(P_dca, residue_labels=residue_labels, coords=coords)
    obj._dca_matrix = np.array(DCA, dtype=np.float64)
    obj._mode       = "dca"
    return obj


@classmethod
def from_dca(
    cls,
    pdb_path: str,
    dca_matrix,
    chain_ids=None,
    cutoff: float = CONTACT_CUTOFF,
    dca_norm_method: str = "minmax",
    floor: float = 1e-3,
    zero_negatives: bool = True,
):
    _, labels, coords = cls._parse_pdb(pdb_path, chain_ids, cutoff)
    DCA = np.array(dca_matrix, dtype=np.float64)
    assert DCA.shape == (len(labels), len(labels)), (
        f"DCA matrix shape {DCA.shape} does not match "
        f"number of PDB residues ({len(labels)}). "
        "Ensure the DCA matrix is aligned to the PDB residue order."
    )
    return cls._from_dca_matrix(
        DCA, labels, coords,
        dca_norm_method=dca_norm_method,
        floor=floor,
        zero_negatives=zero_negatives,
    )


@classmethod
def from_dca_only(
    cls,
    dca_matrix,
    residue_labels=None,
    dca_norm_method: str = "minmax",
    floor: float = 1e-3,
    zero_negatives: bool = True,
):
    if isinstance(dca_matrix, (str,)):
        import pathlib
        if pathlib.Path(dca_matrix).suffix.lower() == ".csv":
            dca_matrix = pd.read_csv(dca_matrix, index_col=0).values
    if isinstance(dca_matrix, pd.DataFrame):
        dca_matrix = dca_matrix.values
    DCA = np.array(dca_matrix, dtype=np.float64)
    assert DCA.ndim == 2 and DCA.shape[0] == DCA.shape[1], f"Expected a square matrix, got shape {DCA.shape}"
    N      = DCA.shape[0]
    labels = residue_labels or [str(i) for i in range(N)]
    assert len(labels) == N, f"residue_labels length {len(labels)} does not match DCA size {N}"
    return cls._from_dca_matrix(
        DCA, labels, coords=None,
        dca_norm_method=dca_norm_method,
        floor=floor,
        zero_negatives=zero_negatives,
    )


def compute_aci_dca(
    self,
    ACTIVE_RESNUMSs,
    n_rounds: int = DEFAULT_N_ROUNDS,
    seed=None,
    dca_threshold: float = 0.0,
):
    if getattr(self, "_mode", None) != "dca":
        raise AttributeError(
            "compute_aci_dca() requires an analyzer built with "
            "OhmAnalyzer.from_dca() or OhmAnalyzer.from_dca_only()."
        )

    P_use = self.P.copy()

    if dca_threshold > 0 and hasattr(self, "_dca_matrix"):
        mask  = self._dca_matrix < dca_threshold
        P_use[mask] = 0.0

    rng    = random.Random(seed)
    N      = self.N
    source = list(ACTIVE_RESNUMSs)
    nbrs   = [np.where(P_use[i] > 0)[0].tolist() for i in range(N)]
    T      = np.zeros(N)

    for _ in range(n_rounds):
        V, W, queue = [0]*N, [0]*N, []
        for s in source:
            V[s] = W[s] = 1; queue.append(s)
        head = 0
        while head < len(queue):
            i = queue[head]; head += 1
            for j in nbrs[i]:
                if not W[j]:
                    W[j] = 1
                    if rng.random() < P_use[i, j]:
                        V[j] = 1
                    queue.append(j)
        for i in range(N):
            T[i] += V[i]

    return T / n_rounds


OhmAnalyzer._from_dca_matrix  = _from_dca_matrix
OhmAnalyzer.from_dca          = from_dca
OhmAnalyzer.from_dca_only     = from_dca_only
OhmAnalyzer.compute_aci_dca   = compute_aci_dca

print("✓ Algorithm D (DCA-driven) patched onto OhmAnalyzer")
print("  New factory methods : from_dca(), from_dca_only()")
print("  New analysis method : compute_aci_dca()")

✓ Algorithm D (DCA-driven) patched onto OhmAnalyzer
  New factory methods : from_dca(), from_dca_only()
  New analysis method : compute_aci_dca()


# **Direct Coupling Analysis - based algorithm.**

Inputs:
1) Direct coupling matrix for the protein derived using the 'create_direct_coupling.ipynb' script.
2) Known active and allosteric residue numbers for the protein.

Outputs:

1.   Allosteric Coupling Intensity ranks for residues (Top 20)
2.   Allosteric Hotspots (Top 5)
3.   Allosteric Pathways (Top 5)
4.   Importance rank for residues (Top 15)


NOTE: This cannot be run for Villin (ref manuscript)

In [ ]:
!pip3 install git+https://github.com/scdantu/pycom

  Cloning https://github.com/scdantu/pycom to /tmp/pip-req-build-0qqcm7h1
  Running command git clone --filter=blob:none --quiet https://github.com/scdantu/pycom /tmp/pip-req-build-0qqcm7h1
  Resolved https://github.com/scdantu/pycom to commit 0674415815db9272d31b05fe15268a7a0875a27e
  Preparing metadata (setup.py) ... done


In [ ]:
import pandas as pd

dca_df = pd.read_csv("Rhodopsin_direct_coupling.csv", index_col=0)
print("Shape:", dca_df.shape)

ACTIVE_RESNUMS = [113, 122, 126, 178, 181, 191, 207, 208, 212, 211,
                        264, 265, 267, 268, 269, 293, 296]
ALLOSTERIC_RESNUMS = [134, 135, 136, 247, 251, 55, 83, 298, 299, 302, 303, 306, 313]
ohm_dca = OhmAnalyzer.from_dca_only(
    dca_matrix=dca_df.values,
    residue_labels=[str(i) for i in range(1, 348)],
)

source_idx     = [ohm_dca.residue_labels.index(str(r)) for r in ACTIVE_RESNUMS]
allosteric_idx = [ohm_dca.residue_labels.index(str(r)) for r in ALLOSTERIC_RESNUMS]

print(f"Source residues     : {ACTIVE_RESNUMS}")
print(f"Source indices      : {source_idx}")
print(f"Allosteric residues : {ALLOSTERIC_RESNUMS}")
print(f"Allosteric indices  : {allosteric_idx}")



N_ROUNDS_DCA = 10_000
SEED_DCA     = 42

aci_dca = ohm_dca.compute_aci(
    ACTIVE_RESNUMSs=source_idx,
    n_rounds=N_ROUNDS_DCA,
    seed=SEED_DCA,
)
print(ohm_dca.aci_report(aci_dca, top_n=20))
print('\n')

pathways_dca = ohm_dca.find_pathways(
    source_idx,
    allosteric_idx,
    n_rounds=N_ROUNDS_DCA,
    seed=SEED_DCA,
)

print('\n')
print(ohm_dca.pathway_report(pathways_dca, top_n=10))
print('\n')
print(ohm_dca.critical_residue_report(pathways_dca, top_n=15))
print('\n')

Shape: (347, 347)
Source residues     : [113, 122, 126, 178, 181, 191, 207, 208, 212, 211, 264, 265, 267, 268, 269, 293, 296]
Source indices      : [112, 121, 125, 177, 180, 190, 206, 207, 211, 210, 263, 264, 266, 267, 268, 292, 295]
Allosteric residues : [134, 135, 136, 247, 251, 55, 83, 298, 299, 302, 303, 306, 313]
Allosteric indices  : [133, 134, 135, 246, 250, 54, 82, 297, 298, 301, 302, 305, 312]
Rank   Residue           ACI
1      269            1.0000
2      296            1.0000
3      122            1.0000
4      208            1.0000
5      181            1.0000
6      191            1.0000
7      178            1.0000
8      211            1.0000
9      265            1.0000
10     267            1.0000
11     207            1.0000
12     264            1.0000
13     212            1.0000
14     293            1.0000
15     268            1.0000
16     126            1.0000
17     113            1.0000
18     109            0.3255
19     112            0.3159
20     295    

# **Dynamic Coupling Analysis - based algorithm.**

Inputs:
1) Dynamic coupling matrix for the protein derived using the 'create_dynamic_coupling.ipynb' script.
2) Known active and allosteric residue numbers for the protein.

Outputs:

1.   Allosteric Coupling Intensity ranks for residues (Top 20)
2.   Allosteric Hotspots (Top 5)
3.   Allosteric Pathways (Top 5)
4.   Importance rank for residues (Top 15)

NOTE: This cannot be run for Rhodopsin (ref manuscript)




In [ ]:
import pandas as pd

dyca_df = pd.read_csv("RAS_dynamic_coupling.csv", index_col=0)
print("Shape:", dyca_df.shape)

ohm_dyca = OhmAnalyzer.from_dca_only(
    dca_matrix=dyca_df.values,
    residue_labels=[str(i) for i in range(1, 87)],
)
"""
ACTIVE_RESNUMS     = [95, 96, 175, 254, 255, 256, 260, 262, 263, 271, 274, 278, 337, 347]
ALLOSTERIC_RESNUMS = [135, 199, 200, 201, 215, 216, 217, 303, 307]
"""

#These are for RAS
ACTIVE_RESNUMS     = [9,47,2,3]
ALLOSTERIC_RESNUMS = [13,7,25,27,31]

source_idx     = [ohm_dyca.residue_labels.index(str(r)) for r in ACTIVE_RESNUMS]
allosteric_idx = [ohm_dyca.residue_labels.index(str(r)) for r in ALLOSTERIC_RESNUMS]

print(f"Source residues     : {ACTIVE_RESNUMS}")
print(f"Source indices      : {source_idx}")
print(f"Allosteric residues : {ALLOSTERIC_RESNUMS}")
print(f"Allosteric indices  : {allosteric_idx}")

N_ROUNDS_dyca = 10_000
SEED_dyca     = 42

aci_dyca = ohm_dyca.compute_aci(
    ACTIVE_RESNUMSs=source_idx,
    n_rounds=N_ROUNDS_dyca,
    seed=SEED_dyca,
)
print(ohm_dyca.aci_report(aci_dyca, top_n=20))

pathways_dyca = ohm_dyca.find_pathways(
    source_idx,
    allosteric_idx,
    n_rounds=N_ROUNDS_dyca,
    seed=SEED_dyca,
)

print('\n')
print(ohm_dyca.pathway_report(pathways_dyca, top_n=10))

print('\n')
print(ohm_dyca.critical_residue_report(pathways_dyca, top_n=15))

Shape: (86, 86)
Source residues     : [9, 47, 2, 3]
Source indices      : [8, 46, 1, 2]
Allosteric residues : [13, 7, 25, 27, 31]
Allosteric indices  : [12, 6, 24, 26, 30]
Rank   Residue           ACI
1      9              1.0000
2      47             1.0000
3      2              1.0000
4      3              1.0000
5      10             0.7747
6      8              0.6997
7      79             0.1039
8      81             0.0968
9      6              0.0926
10     11             0.0802
11     7              0.0801
12     53             0.0745
13     52             0.0734
14     4              0.0708
15     80             0.0671
16     61             0.0637
17     5              0.0634
18     76             0.0612
19     50             0.0584
20     51             0.0574


Pathway                                                             Weight
Start -> 9 -> 8 -> 7 -> End                                         0.0067
Start -> 2 -> 7 -> End                                             

#**Comparison of performance across all algorithms.**

The script belowscipt compares the ranks of residue importance and ACI for of the actual allosteric sites in order to evaluate performance of the algorithms.

**NOTE:** Since not all algorithms can be run for all proteins (limited by the availability of input data), it is recommended to comment out the key for the inapplicable algorithm in the 'METHODS' dictionary.

In [ ]:
ACTIVE_RESNUMS = [95, 96, 175, 254, 255, 256, 260, 262, 263, 271, 274, 278, 337, 347]
ALLOSTERIC_RESNUMS = [135, 199, 200, 201, 215, 216, 217, 303, 307, 370, 374]

ohm = OhmAnalyzer.from_pdb("Rhodopsin.pdb")
active_idx = get_idx(ohm, ACTIVE_RESNUMS)
aci = ohm.compute_aci(active_idx, n_rounds=N_ROUNDS, seed=SEED)

METHODS = {
    "Struct":  dict(ohm=ohm,      aci=aci,      pathways=pathways),
    "TC":      dict(ohm=ohm_tc,   aci=aci_tc,   pathways=pathways_tc),
    "TC-Filt": dict(ohm=ohm_c,    aci=aci_c,    pathways=pathways_c),
    "DCA":     dict(ohm=ohm_dca,  aci=aci_dca,  pathways=pathways_dca),
    #"DyCA":    dict(ohm=ohm_dyca, aci=aci_dyca, pathways=pathways_dyca),
}

active_methods = {}
for name, deps in METHODS.items():
    try:
        assert all(v is not None for v in deps.values())
        active_methods[name] = deps
    except (NameError, AssertionError):
        print(f"[skip] {name} — missing dependencies")

aci_ranks = {
    name: {idx: rank + 1 for rank, idx in enumerate(np.argsort(deps["aci"])[::-1])}
    for name, deps in active_methods.items()
}

ref_ohm = next(iter(active_methods.values()))["ohm"]
header_cols = "".join(f"{name:>10}" for name in active_methods)
print("ACI RANKS")
print(f"{'Residue':<12}{header_cols}")

for i, label in enumerate(ref_ohm.residue_labels):
    if int(label.split("/")[1]) in ALLOSTERIC_RESNUMS:
        row = "".join(f"{aci_ranks[name][i]:>10}" for name in active_methods)
        print(f"{label:<12}{row}")

imp = {
    name: deps["ohm"].critical_residues(deps["pathways"])
    for name, deps in active_methods.items()
}

all_residues = sorted(set().union(*imp.values()))

def make_rank(imp_dict, all_res):
    scores = {r: imp_dict.get(r, 0.0) for r in all_res}
    return {r: rank + 1 for rank, r in
            enumerate(sorted(scores, key=scores.__getitem__, reverse=True))}

imp_ranks = {name: make_rank(imp[name], all_residues) for name in active_methods}

print("\nIMPORTANCE RANKS")
print(f"{'Residue':<12}{header_cols}")

for r in all_residues:
    label  = ref_ohm.residue_labels[r]
    resnum = int(label.split("/")[1])
    if resnum not in ALLOSTERIC_RESNUMS:
        continue
    row = "".join(f"{imp_ranks[name][r]:>10}" for name in active_methods)
    print(f"{label:<12}{row}")

Saved: Rhodopsin_contact_matrix_raw.npy / .csv
Saved: Rhodopsin_contact_matrix_norm.npy / .csv
ACI RANKS
Residue         Struct        TC   TC-Filt       DCA
A/135               79       200        57       256
A/199              237       167       299       279
A/200              188       120        59        68
A/201              197        27        69       122
A/215              218       233       348       190
A/216               36        57        36       158
A/217               52       152       205       265
A/303              222       161       209       123
A/307              243       191       210       232

IMPORTANCE RANKS
Residue         Struct        TC   TC-Filt       DCA
A/135               34        67        34        24
A/199              234       173       234        73
A/200              123       206       123       282
A/201              179       168       179       300
A/215               37        42        37       100
A/216               24       